In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
from trend_analisys_rsi_markov import TrendAnalyzer
import warnings
from scripts.assetsRoster import carteira_AC, carteira_HB, others
warnings.filterwarnings('ignore')


In [2]:
asset_ids = [
    'bitcoin', 'ethereum', 'binancecoin', 'ripple', 'cardano',
    'solana', 'polkadot', 'avalanche-2', 'chainlink', 'uniswap',
    'heyanon'
]

In [3]:
data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/'
btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv'
ssr_data_path='/Users/valter.rebelo/MissionControl/data/onchainData/BTC_SSR.csv'

In [17]:
analyzer = TrendAnalyzer(
    asset_ids=set(carteira_HB+carteira_AC),
    data_path=data_path,
    btc_data_path=btc_data_path,
    ssr_data_path=ssr_data_path,
    use_btc_adjusted=True,
    verbose=True,
    lookback_days=180,
    trend_metrics_lookback=365,
    markov_model_name='20250325_144631'
)

2025-03-26 20:28:41,198 - INFO - Loaded Markov volatility model: 20250325_144631
2025-03-26 20:28:41,205 - INFO - Successfully loaded SSR data with 2353 rows.


Data loaded successfully: 4102 records
Training set: 3281 records
Test set: 821 records
Model metadata:
  Saved on: 2025-03-25
  Data range: 2014-01-01 to 2025-03-24
  Records: 4101 total, 3075 train, 1026 test
Model loaded from /Users/valter.rebelo/MissionControl/models/markov_volatility_model_20250325_144631.pkl


In [ ]:
btc_data = analyzer.asset_data.get('bitcoin').get('classified_data')
# Create a copy of the full date range
all_dates = btc_data[btc_data['date'] >= '2019-01-01'].copy()
# Create a mask for the condition
condition_mask = (#((all_dates['Short Term (BTC)'] == 'Strong Bull') | (all_dates['Short Term (BTC)'] == 'Weak Bull')) 
                 
#                 &

                 ((all_dates['Overall (USD)'] == 'Strong Bull') | (all_dates['Overall (USD)'] == 'Weak Bull'))
                 
                 &

                 ((all_dates['Short Term (USD)'] == 'Strong Bull')))

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create the plotly figure
fig = go.Figure()

# Add a base line trace with the complete price history (in red by default)
fig.add_trace(go.Scatter(
    x=all_dates['date'],
    y=all_dates['close'],
    mode='lines',
    name='Non-matching conditions',
    line=dict(color='red', width=2)
))

# Add green trace only for the matching conditions
# Create a copy with NaN values for non-matching points
matching_data = all_dates.copy()
matching_data.loc[~condition_mask, 'close'] = np.nan

fig.add_trace(go.Scatter(
    x=matching_data['date'],
    y=matching_data['close'],
    mode='lines',
    name='Matching conditions',
    line=dict(color='green', width=2),
    connectgaps=False
))

# Update layout
fig.update_layout(
    title='Classificação de Tendência: BTC',
    xaxis_title='Date',
    yaxis_title='Close Price (USD)',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified'
)

# Add grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

# Show the plot
fig.show()

# Make sure every date is classified in one of the two categories
total_points = len(all_dates)
matching_points = condition_mask.sum()
non_matching_points = (~condition_mask).sum()

# Verify that all points are classified
if matching_points + non_matching_points != total_points:
    print(f"Warning: {total_points - (matching_points + non_matching_points)} points were not classified")

In [ ]:
btc_data = analyzer.asset_data.get('ripple').get('classified_data')
# Create a copy of the full date range
all_dates = btc_data[btc_data['date'] >= '2019-01-01'].copy()
# Create a mask for the condition
condition_mask = (((all_dates['Short Term (BTC)'] == 'Strong Bull') | (all_dates['Short Term (BTC)'] == 'Weak Bull')) 
                 
#                 &

                # ((all_dates['Overall (BTC)'] == 'Strong Bull'))
                 
                 &

                 ((all_dates['Short Term (USD)'] == 'Strong Bull')))    

import numpy as np
import plotly.graph_objects as go

# Create two separate dataframes for plotting
matching_data = all_dates.copy()
non_matching_data = all_dates.copy()

# For matching conditions: keep only matching points, set others to NaN
matching_data.loc[~condition_mask, 'close'] = np.nan

# For non-matching conditions: keep only non-matching points, set others to NaN
non_matching_data.loc[condition_mask, 'close'] = np.nan

# Create plotly figure
fig = go.Figure()

# Add traces for matching and non-matching conditions
fig.add_trace(go.Scatter(
    x=matching_data['date'],
    y=matching_data['close'],
    mode='lines',
    name='Matching conditions',
    line=dict(color='green', width=2),
    connectgaps=False
))

fig.add_trace(go.Scatter(
    x=non_matching_data['date'],
    y=non_matching_data['close'],
    mode='lines',
    name='Non-matching conditions',
    line=dict(color='red', width=2),
    connectgaps=False
))

# Update layout
fig.update_layout(
    title='Classificação de Tendência: XRP',
    xaxis_title='Date',
    yaxis_title='Close Price (USD)',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified'
)

# Add grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

# Show the plot
fig.show()

# Make sure every date is classified in one of the two categories
total_points = len(all_dates)
matching_points = matching_data['close'].notna().sum()
non_matching_points = non_matching_data['close'].notna().sum()

# Verify that all points are classified
if matching_points + non_matching_points != total_points:
    print(f"Warning: {total_points - (matching_points + non_matching_points)} points were not classified")

In [18]:

# Analyze assets
analyzer.analyze_multiple_assets()


Computing lookback metrics: 100%|██████████| 48/48 [00:01<00:00, 26.04it/s]
2025-03-26 20:30:19,829 - INFO - Trend analysis completed for multiple assets.


,Ticker,Short Term Trend (USD),Medium Term Trend (USD),Long Term Trend (USD),Overall Trend (USD),Short Term Trend (BTC),Medium Term Trend (BTC),Long Term Trend (BTC),Overall Trend (BTC),Sharpe Ratio (USD),...,Current Trend Sortino (USD),Current Trend Mean Return (USD),Current Trend Median Return (USD),Current Trend Skew (USD),Current Trend Sharpe (BTC),Current Trend Sortino (BTC),Current Trend Mean Return (BTC),Current Trend Median Return (BTC),Current Trend Skew (BTC),Latest Date
0,MUBI,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Weak Bull,Strong Bear,Strong Bear,Weak Bear,0.401802,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
1,UNI,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Weak Bull,Strong Bear,Strong Bear,Weak Bear,0.697793,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
2,GMX,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,0.589390,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
3,SCRT,Strong Bull,Strong Bear,Weak Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,0.509310,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
4,MATIC,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,1.115569,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
5,RUNE,Strong Bull,Weak Bull,Strong Bear,Weak Bull,Strong Bear,Weak Bull,Strong Bear,Weak Bear,1.206054,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
6,ETH,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,1.178232,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
7,SAND,Strong Bull,Strong Bear,Weak Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,0.894484,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
8,FXS,Strong Bull,Strong Bull,Weak Bear,Weak Bull,Strong Bull,Strong Bull,Strong Bear,Weak Bull,0.573020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25
9,SOL,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,1.422233,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-03-25


In [131]:
start_date='2024-01-1'
end_date='2025-03-25'

# Create a BTC-only trend-following portfolio
analyzer.create_portfolio(
    portfolio_name='btc_trend_following',
    #usd_conditions={'Short Term (USD)': ['Weak Bull','Strong Bull']},
    btc_only=True,
    btc_trend_gating=None,
    rsi_conditions_usd=True,
    use_ssr_signal=True,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='big_wins_i',
    btc_conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=False,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='big_wins_ii',
    btc_conditions={'Short Term (BTC)': ['Weak Bull','Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=True,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)


# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='fat_tails',
    btc_conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Weak Bull','Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=True,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='experiment_i',
    btc_conditions={'Short Term (BTC)': ['Weak Bull','Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bear','Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=False,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)


########################################################

# Backtest the altcoin portfolio
results_alt = analyzer.backtest_portfolio(
    portfolio_name='big_wins_i',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.005,
    signal_threshold=75
)

results_alt = analyzer.backtest_portfolio(
    portfolio_name='experiment_i',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.005,
    signal_threshold=75
)

# Backtest the BTC trend-following portfolio
results_btc = analyzer.backtest_portfolio(
    portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.001,
    signal_threshold=50
)

2025-03-27 11:42:55,619 - WARNING - Portfolio btc_trend_following already exists. Overwriting.
2025-03-27 11:42:55,626 - INFO - Portfolio 'btc_trend_following' created with criteria: BTC gating=None, USD conditions=None, BTC conditions=None, RSI USD=True, RSI BTC=False, volatility filter=True, volatility weight=1, SSR signal=True, SSR gate=False, BTC RSI gate=False, BTC RSI signal=False, Follow portfolio=None, BTC only=True
2025-03-27 11:42:55,628 - WARNING - Portfolio big_wins_i already exists. Overwriting.
2025-03-27 11:42:55,632 - INFO - Portfolio 'big_wins_i' created with criteria: BTC gating=None, USD conditions={'Short Term (USD)': ['Strong Bull']}, BTC conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Strong Bull']}, RSI USD=True, RSI BTC=True, volatility filter=True, volatility weight=1, SSR signal=False, SSR gate=False, BTC RSI gate=False, BTC RSI signal=False, Follow portfolio=btc_trend_following, BTC only=False
2025-03-27 11:42:55,633 - WARNING - Portfolio 

In [132]:


# Plot performance for a specific date range
fig = analyzer.plot_individual_asset_performance(
    portfolio_name='experiment_i',
    btc_trend_portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    show_plot=False,
   # asset_filter=['SOL']
)


2025-03-27 11:43:27,146 - INFO - Recalculating backtest for portfolio 'experiment_i' from 2024-01-1 to 2025-03-25.
2025-03-27 11:43:27,167 - INFO - Using signals from BTC-only portfolio 'btc_trend_following'


2025-03-27 11:43:42,338 - INFO - Recalculating backtest for portfolio 'btc_trend_following' from 2024-01-1 to 2025-03-25.
2025-03-27 11:43:42,350 - INFO - Entering BTC position on 2024-01-08 00:00:00: Cost = 10.00
2025-03-27 11:43:42,353 - INFO - Exiting BTC position on 2024-01-09 00:00:00: Cost = 9.99
2025-03-27 11:43:42,354 - INFO - Entering BTC position on 2024-01-10 00:00:00: Cost = 9.98
2025-03-27 11:43:42,359 - INFO - Exiting BTC position on 2024-01-13 00:00:00: Cost = 9.29
2025-03-27 11:43:42,361 - INFO - Entering BTC position on 2024-01-14 00:00:00: Cost = 9.28
2025-03-27 11:43:42,364 - INFO - Exiting BTC position on 2024-01-16 00:00:00: Cost = 9.24
2025-03-27 11:43:42,379 - INFO - Entering BTC position on 2024-01-30 00:00:00: Cost = 9.23
2025-03-27 11:43:42,405 - INFO - Exiting BTC position on 2024-02-27 00:00:00: Cost = 11.63
2025-03-27 11:43:42,407 - INFO - Entering BTC position on 2024-02-28 00:00:00: Cost = 11.62
2025-03-27 11:43:42,424 - INFO - Exiting BTC position on 202


Performance Metrics:
BTC Buy & Hold: Return = 98.15%, Max Drawdown = -26.23%
BTC Trend (btc_trend_following): Return = 99.00%, Max Drawdown = -19.76%

Individual Asset Performance:

|         | Total Return   | Max Drawdown   |   Sharpe Ratio |   Sortino Ratio |   Number of Trades | Win Rate   | Avg Win   | Avg Loss   |   Avg Holding Days | Total Costs   | First Trade         | Last Trade          |   Trading Days |
|:--------|:---------------|:---------------|---------------:|----------------:|-------------------:|:-----------|:----------|:-----------|-------------------:|:--------------|:--------------------|:--------------------|---------------:|
| VIRTUAL | 10685.58%      | -63.81%        |           7.77 |           22.45 |                 19 | 42.11%     | 197.48%   | -11.66%    |                7.5 | $99,765.09    | 2024-02-06 00:00:00 | 2025-01-19 00:00:00 |            142 |
| AERO    | 368.20%        | -63.30%        |           4.45 |           13.21 |                 16 | 3

In [124]:
# Plot performance for a specific date range
fig = analyzer.plot_individual_asset_performance(
    portfolio_name='big_wins_i',
    btc_trend_portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    show_plot=False,
   # asset_filter=['SOL']
)

2025-03-27 10:50:11,003 - INFO - Recalculating backtest for portfolio 'big_wins_i' from 2024-01-1 to 2025-03-25.
2025-03-27 10:50:11,102 - INFO - Using signals from BTC-only portfolio 'btc_trend_following'
2025-03-27 10:50:27,068 - INFO - Recalculating backtest for portfolio 'btc_trend_following' from 2024-01-1 to 2025-03-25.
2025-03-27 10:50:27,090 - INFO - Entering BTC position on 2024-01-08 00:00:00: Cost = 10.00
2025-03-27 10:50:27,092 - INFO - Exiting BTC position on 2024-01-09 00:00:00: Cost = 9.99
2025-03-27 10:50:27,094 - INFO - Entering BTC position on 2024-01-10 00:00:00: Cost = 9.98
2025-03-27 10:50:27,100 - INFO - Exiting BTC position on 2024-01-13 00:00:00: Cost = 9.29
2025-03-27 10:50:27,102 - INFO - Entering BTC position on 2024-01-14 00:00:00: Cost = 9.28
2025-03-27 10:50:27,107 - INFO - Exiting BTC position on 2024-01-16 00:00:00: Cost = 9.24
2025-03-27 10:50:27,126 - INFO - Entering BTC position on 2024-01-30 00:00:00: Cost = 9.23
2025-03-27 10:50:27,157 - INFO - Exit


Performance Metrics:
BTC Buy & Hold: Return = 98.15%, Max Drawdown = -26.23%
BTC Trend (btc_trend_following): Return = 99.00%, Max Drawdown = -19.76%

Individual Asset Performance:

|         | Total Return   | Max Drawdown   |   Sharpe Ratio |   Sortino Ratio |   Number of Trades | Win Rate   | Avg Win   | Avg Loss   |   Avg Holding Days | Total Costs   | First Trade         | Last Trade          |   Trading Days |
|:--------|:---------------|:---------------|---------------:|----------------:|-------------------:|:-----------|:----------|:-----------|-------------------:|:--------------|:--------------------|:--------------------|---------------:|
| VIRTUAL | 3636.93%       | -68.94%        |          10.77 |           28.59 |                 19 | 36.84%     | 192.57%   | -13.10%    |                6.2 | $36,822.64    | 2024-02-06 00:00:00 | 2025-01-17 00:00:00 |            118 |
| AERO    | 404.81%        | -64.89%        |           5.88 |           16.88 |                 17 | 4

In [ ]:
def get_signals_for_period(start_date, end_date, assets='all'):
    results_df = analyzer.get_portfolio_details(portfolio_name='altcoins_trend_following_btc').get('backtest_results').get('signals_df')
    
    # Filter by date range
    signals = results_df[(results_df.index >= start_date) & (results_df.index <= end_date)]
    
    # Filter by assets if specified
    if assets != 'all':
        if isinstance(assets, str):
            assets = [assets]
        signals = signals[signals['asset'].isin(assets)]
        
    return signals

# Example usage:
start_date = '2024-7-31'
end_date = '2025-3-25' 

assets = ['ETH']  # or 'all' for all assets
#assets = 'all'
signals = get_signals_for_period(start_date, end_date, assets)
signals[(signals['followed_portfolio_signal'] == 1) & (signals['final_decision'] == 0)]

In [ ]:
assets_held = analyzer.get_portfolio_details(portfolio_name='btc_gated_rsi_vol_momentum').get('backtest_results').get('results_df')
assets_held

import matplotlib.pyplot as plt

# Get the results dataframe

# Create the plot
plt.figure(figsize=(14, 6))

# Plot the assets held line
plt.plot(assets_held.index, assets_held['Assets_Held'], 'b-')

# Add vertical lines for each month
for date in assets_held.index[assets_held.index.is_month_start]:
    plt.axvline(x=date, color='gray', alpha=0.3, linestyle='--')

plt.title('Assets Held Over Time')
plt.ylabel('Number of Assets')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
test_criteria = [{'Short Term (USD)': ['Strong Bull'],
                  'Short Term (BTC)': ['Strong Bull'],
                  'Overall (BTC)': ['Strong Bull']},
                  
                 {'Short Term (BTC)': ['Strong Bull'],
                  'Short Term (USD)': ['Weak Bull', 'Strong Bull']}]

In [ ]:
btc_data = analyzer.asset_data['maker'].get('classified_data')[['date', 'Short Term (USD)', 'Overall (USD)', 'RSI_Signal_close']]
btc_data[btc_data['date'] == data]


In [33]:
results_df = analyzer.get_portfolio_details(portfolio_name='altcoins_trend_following_btc').get('backtest_results')
results_df

{'results_df': Empty DataFrame
 Columns: []
 Index: [2024-01-01 00:00:00, 2024-01-02 00:00:00, 2024-01-03 00:00:00, 2024-01-04 00:00:00, 2024-01-05 00:00:00, 2024-01-06 00:00:00, 2024-01-07 00:00:00, 2024-01-08 00:00:00, 2024-01-09 00:00:00, 2024-01-10 00:00:00, 2024-01-11 00:00:00, 2024-01-12 00:00:00, 2024-01-13 00:00:00, 2024-01-14 00:00:00, 2024-01-15 00:00:00, 2024-01-16 00:00:00, 2024-01-17 00:00:00, 2024-01-18 00:00:00, 2024-01-19 00:00:00, 2024-01-20 00:00:00, 2024-01-21 00:00:00, 2024-01-22 00:00:00, 2024-01-23 00:00:00, 2024-01-24 00:00:00, 2024-01-25 00:00:00, 2024-01-26 00:00:00, 2024-01-27 00:00:00, 2024-01-28 00:00:00, 2024-01-29 00:00:00, 2024-01-30 00:00:00, 2024-01-31 00:00:00, 2024-02-01 00:00:00, 2024-02-02 00:00:00, 2024-02-03 00:00:00, 2024-02-04 00:00:00, 2024-02-05 00:00:00, 2024-02-06 00:00:00, 2024-02-07 00:00:00, 2024-02-08 00:00:00, 2024-02-09 00:00:00, 2024-02-10 00:00:00, 2024-02-11 00:00:00, 2024-02-12 00:00:00, 2024-02-13 00:00:00, 2024-02-14 00:00:00, 20

In [35]:
signals_btc = analyzer.get_portfolio_details(portfolio_name='alt_short_momentum_low_vol').get('backtest_results').get('results_df')
signals_btc.head(60)

""
2024-01-01
2024-01-02
2024-01-03
2024-01-04
2024-01-05
2024-01-06
2024-01-07
2024-01-08
2024-01-09
2024-01-10


In [ ]:
analyzer.asset_data.get('maker')